# Glider / CTD Map App — Barkley Sound

Interactive Leaflet basemap for browsing glider tracks and CTD casts spatially, instead of
scrolling through notebook cells looking for the right dataset. Every marker/line on the map is
clickable and pops up the *same* plot `Glider_Curtain_Plot.ipynb` would produce for that dataset
(reused, not reimplemented, from `glider_lib.py` — see that file's docstring for how the two
notebooks stay in sync).

- **CTD casts** → a single marker at the cast's (lon, lat). Click it → 2D profile popup
  (`plot_ctd_profile`).
- **Glider tracks** → a polyline over the surfaced lon/lat path. Click it → 3D curtain popup
  (`plot_glider_curtain`).
- **Basemap** → Esri Ocean Basemap, framed to the region's bounding box from
  `Glider_Curtain_Plot.ipynb`'s `CONFIG["REGION"]`.

This uses **ipyleaflet** (a real Leaflet.js map wired to live Python callbacks via Jupyter
widgets), not a static HTML map — clicking a layer re-runs a plotting function against that
layer's data right now, in this kernel. See the last section for how to launch this outside of
JupyterLab as a standalone app (Voila).

**Roadmap** (not built yet, noted here so the structure below stays extensible):
- Long-term mooring time series as another clickable layer type.
- Full-grid netCDF current fields (e.g. modeled u/v velocity) as a vector-field overlay on the
  same map — planned as either periodic quiver/streamline raster tiles or a dedicated Leaflet
  velocity plugin, added as one more `ipyleaflet.Layer` alongside the marker/polyline layers
  below, so it composes with everything already here rather than needing a separate map.

## 1. Imports

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ipyleaflet import (
    Map, Marker, Polyline, Popup, basemaps,
    LayersControl, ScaleControl, FullScreenControl,
)
from ipywidgets import Output

from glider_lib import (
    load_platform_data,
    generate_sample_glider_data,
    plot_ctd_profile,
    plot_glider_curtain,
)

## 2. Configuration

Mirrors the relevant parts of `Glider_Curtain_Plot.ipynb`'s `CONFIG` cell — same `REGION`
bounding box and dataset paths/column maps. Kept as a separate dict here (rather than imported
from the other notebook) since `.ipynb` files aren't directly importable; if `CONFIG` in the
curtain notebook changes, update it here too.

In [7]:
CONFIG_MAP = {
    "REGION": {"lon_range": (-126.8, -124.5), "lat_range": (47.85, 49.36)},  # Barkley Sound, BC

    "CTD": {
        "DATA_PATH": "NE_San_Diego_Trough_Aug_2022.csv",
        "FILE_TYPE": "csv",
        "COLUMN_MAP": {"lon": None, "lat": None, "depth": None, "variable": "Salt2"},
        "VARIABLE_LABEL": "Salinity (PSU)",
        "LINE_COLOR": "#1b6ca8",
        "DEPTH_POSITIVE_DOWN": True,
        "MARKER_COLOR": "#d1495b",
    },

    "GLIDER": {
        "DATA_PATH": "path/to/glider_data.csv",  # not yet configured -- falls back to sample data below
        "FILE_TYPE": "csv",
        "COLUMN_MAP": {"lon": None, "lat": None, "depth": None, "variable": "Temperature"},
        "VARIABLE_LABEL": "Temperature (°C)",
        "COLOR_SCALE": "Thermal",
        "DEPTH_POSITIVE_DOWN": True,
        "LINE_COLOR": "#f4a261",
    },
}

# Popup plots are rendered small so they read as a preview, not a full dashboard panel.
POPUP_FIGURE_SIZE = dict(width=380, height=320)

## 3. Load CTD + glider data

Real loader for the CTD cast (same `load_platform_data()` used by the curtain notebook). The
glider path in `CONFIG_MAP` is still the placeholder from `Glider_Curtain_Plot.ipynb` — falls
back to a synthetic sample track so the map isn't empty; swap in a real `DATA_PATH` +
`COLUMN_MAP` once a glider file is available and this cell picks it up automatically.

In [8]:
ctd_cfg = CONFIG_MAP["CTD"]
ctd_var = ctd_cfg["COLUMN_MAP"]["variable"]
ctd_df = load_platform_data(ctd_cfg["DATA_PATH"], ctd_cfg["FILE_TYPE"], ctd_cfg["COLUMN_MAP"])
if ctd_cfg["DEPTH_POSITIVE_DOWN"]:
    ctd_df["Depth"] = -ctd_df["Depth"].abs()
print(f"CTD cast: {len(ctd_df)} rows at ({ctd_df['Longitude'].iloc[0]:.3f}, {ctd_df['Latitude'].iloc[0]:.3f})")

glider_cfg = CONFIG_MAP["GLIDER"]
glider_var = glider_cfg["COLUMN_MAP"]["variable"]
try:
    glider_df = load_platform_data(glider_cfg["DATA_PATH"], glider_cfg["FILE_TYPE"], glider_cfg["COLUMN_MAP"])
    print(f"Glider: {len(glider_df)} rows loaded from {glider_cfg['DATA_PATH']}")
except FileNotFoundError:
    glider_df = generate_sample_glider_data(variable_col=glider_var, **CONFIG_MAP["REGION"])
    print(f"Glider: '{glider_cfg['DATA_PATH']}' not found -- showing {len(glider_df)}-point sample "
          f"track instead. Set CONFIG_MAP['GLIDER']['DATA_PATH'] to a real file to replace it.")
if glider_cfg["DEPTH_POSITIVE_DOWN"]:
    glider_df["Depth"] = -glider_df["Depth"].abs()

# Sanity check against REGION -- a mismatch here just means the initial map view won't be
# centered on this dataset (you can still pan/zoom to it); it doesn't block anything below.
lon_lo, lon_hi = CONFIG_MAP["REGION"]["lon_range"]
lat_lo, lat_hi = CONFIG_MAP["REGION"]["lat_range"]
for name, df in [("CTD", ctd_df), ("glider", glider_df)]:
    out_of_region = not ((lon_lo <= df["Longitude"]).all() and (df["Longitude"] <= lon_hi).all()
                          and (lat_lo <= df["Latitude"]).all() and (df["Latitude"] <= lat_hi).all())
    if out_of_region:
        print(f"  ⚠ {name} data falls outside CONFIG_MAP['REGION']'s bounding box "
              f"(lon [{lon_lo}, {lon_hi}], lat [{lat_lo}, {lat_hi}]) -- the map will still frame "
              "REGION on load, but you'll need to pan to see this layer.")

CTD cast: 516 rows at (-117.532, 32.845)
Glider: 'path/to/glider_data.csv' not found -- showing 500-point sample track instead. Set CONFIG_MAP['GLIDER']['DATA_PATH'] to a real file to replace it.
  ⚠ CTD data falls outside CONFIG_MAP['REGION']'s bounding box (lon [-126.8, -124.5], lat [47.85, 49.36]) -- the map will still frame REGION on load, but you'll need to pan to see this layer.


## 4. Build the basemap

Framed to `CONFIG_MAP["REGION"]` via `fit_bounds`, exactly as requested — the initial view is
the region's bounding box, independent of where any loaded dataset actually falls (see the
warning above if they don't overlap).

In [9]:
m = Map(
    basemap=basemaps.Esri.OceanBasemap,
    center=((lat_lo + lat_hi) / 2, (lon_lo + lon_hi) / 2),
    zoom=8,
    scroll_wheel_zoom=True,
)
m.fit_bounds([[lat_lo, lon_lo], [lat_hi, lon_hi]])
m.add(LayersControl(position="topright"))
m.add(ScaleControl(position="bottomleft"))
m.add(FullScreenControl())
m

Map(center=[48.605000000000004, -125.65], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_…

## 5. Click-to-plot layers

Each layer's `on_click` handler builds the popup lazily, the first time that layer is clicked
(not up front for every dataset) — cheap for a couple of layers now, and keeps this cheap once
there are many.

In [5]:
def make_popup_handler(build_figure, location):
    '''Returns an on_click handler that lazily builds a Plotly-in-Popup the first time it fires.'''
    state = {"popup": None}

    def handler(**kwargs):
        if state["popup"] is None:
            fig = build_figure()
            fig.update_layout(**POPUP_FIGURE_SIZE, margin=dict(l=40, r=10, t=40, b=40))
            output = Output()
            with output:
                from IPython.display import display
                display(fig)
            state["popup"] = Popup(location=location, child=output, close_button=True, auto_close=False)
        if state["popup"] not in m.layers:
            m.add(state["popup"])

    return handler


# --- CTD marker ---
ctd_location = (ctd_df["Latitude"].iloc[0], ctd_df["Longitude"].iloc[0])
ctd_marker = Marker(location=ctd_location, title="CTD cast", draggable=False)
ctd_marker.on_click(make_popup_handler(
    lambda: plot_ctd_profile(ctd_df, ctd_var, variable_label=ctd_cfg["VARIABLE_LABEL"],
                              line_color=ctd_cfg["LINE_COLOR"]),
    ctd_location,
))
m.add(ctd_marker)

# --- Glider track ---
glider_locations = list(zip(glider_df["Latitude"], glider_df["Longitude"]))
glider_line = Polyline(locations=glider_locations, color=glider_cfg["LINE_COLOR"], weight=3, fill=False)
glider_midpoint = glider_locations[len(glider_locations) // 2]
glider_line.on_click(make_popup_handler(
    lambda: plot_glider_curtain(glider_df, glider_var, variable_label=glider_cfg["VARIABLE_LABEL"],
                                 color_scale=glider_cfg["COLOR_SCALE"]),
    glider_midpoint,
))
m.add(glider_line)

print("Click the CTD marker or the glider track on the map above to pop up its plot.")

Click the CTD marker or the glider track on the map above to pop up its plot.


## 6. Launching this as a standalone app

Right now this runs inline in JupyterLab, which is enough to use it interactively today. To
launch it as its own browser tab without the notebook/code chrome (a real "app"), serve it with
[Voila](https://voila.readthedocs.io/) — it re-executes this notebook and shows only the
rendered widgets, with the same live click → Python → popup behavior:

```bash
pip install voila   # not installed in this environment yet
voila Glider_Map_App.ipynb
```

On this JupyterHub, `jupyter-server-proxy` is already installed, so a running Voila server is
reachable at `<hub-url>/user/<your-username>/voila/render/final_notebooks/Glider_Map_App.ipynb`
without opening any extra ports — that URL is shareable with anyone who can reach the hub.